# Seq2Seq and Attention

## Learning Objectives
1. Compute Bahdanau attention scores and context vectors in numpy.
2. Build a full seq2seq model with additive attention in PyTorch.
3. Visualize attention alignment and diagnose exposure bias.
4. Compare no-attention vs additive vs dot-product attention on a reversal task.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## Level 1: Bahdanau Attention in Numpy

Given a decoder query s (dim=8) and encoder hidden states h (5 timesteps × dim=8):

```
energy_j = v^T  tanh(W1 @ s  +  W2 @ h_j)    for j in 1..T
alpha    = softmax(energy)                      attention weights
context  = sum_j  alpha_j * h_j                 weighted sum of encoder states
```

The alignment model is a small feedforward net learned jointly with the seq2seq model.

In [ ]:
# ── Dimensions ────────────────────────────────────────────────────────
T_ENC   = 5    # encoder sequence length (5 source tokens)
DIM_H   = 8    # encoder/decoder hidden dimension
DIM_ATT = 8    # attention projection dimension

rng = np.random.default_rng(0)

# Encoder hidden states: h[j] for j in 0..T_ENC-1
h_enc = rng.normal(0, 0.5, (T_ENC, DIM_H))   # [5, 8]
# Decoder query (current decoder hidden state)
s_dec = rng.normal(0, 0.5, DIM_H)             # [8]

# Bahdanau alignment parameters
W1  = rng.normal(0, 0.1, (DIM_ATT, DIM_H))   # project decoder query: [8, 8]
W2  = rng.normal(0, 0.1, (DIM_ATT, DIM_H))   # project encoder states: [8, 8]
v   = rng.normal(0, 0.1, DIM_ATT)             # scoring vector: [8]

def softmax_np(x: np.ndarray) -> np.ndarray:
    """Numerically stable softmax."""
    e = np.exp(x - x.max())
    return e / e.sum()

def bahdanau_attention(s: np.ndarray, h: np.ndarray,
                       W1: np.ndarray, W2: np.ndarray,
                       v: np.ndarray) -> tuple:
    """
    Args:
        s:  decoder query        [H]
        h:  encoder hiddens      [T, H]
        W1: decoder projection   [A, H]
        W2: encoder projection   [A, H]
        v:  scoring vector       [A]
    Returns:
        context: weighted sum of encoder states [H]
        alpha:   attention weights              [T]
    """
    # Project decoder query once (same for all positions)
    q_proj = W1 @ s                      # [A]
    energies = np.zeros(len(h))          # [T]

    for j in range(len(h)):
        # Project encoder state j and combine with decoder query
        k_proj  = W2 @ h[j]              # [A]
        energy  = v @ np.tanh(q_proj + k_proj)  # scalar
        energies[j] = energy

    alpha   = softmax_np(energies)       # [T]  sum to 1
    context = (alpha[:, None] * h).sum(0)  # [H]  weighted encoder states
    return context, alpha

context, alpha = bahdanau_attention(s_dec, h_enc, W1, W2, v)

print(f'Attention weights alpha: {alpha.round(3)}')
print(f'Sum of alpha:            {alpha.sum():.4f}  (must be 1.0)')
print(f'Context vector shape:    {context.shape}')
print(f'Context vector:          {context.round(3)}')

# ── Vectorised version (batch-friendly) ──────────────────────────────
def bahdanau_vectorised(s: np.ndarray, h: np.ndarray,
                        W1: np.ndarray, W2: np.ndarray,
                        v: np.ndarray) -> tuple:
    """Batch-efficient vectorised Bahdanau attention."""
    q_proj = W1 @ s              # [A]
    k_proj = (W2 @ h.T).T        # [T, A]
    combined = np.tanh(q_proj[None, :] + k_proj)   # [T, A]
    energies = combined @ v      # [T]
    alpha_v  = softmax_np(energies)
    context_v = (alpha_v[:, None] * h).sum(0)
    return context_v, alpha_v

ctx_v, alp_v = bahdanau_vectorised(s_dec, h_enc, W1, W2, v)
print(f'\nVectorised matches loop: '
      f'{np.allclose(ctx_v, context, atol=1e-6) and np.allclose(alp_v, alpha, atol=1e-6)}')

# ── Bar chart of attention weights ────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(T_ENC), alpha, color='steelblue', edgecolor='white')
ax.set_xlabel('Encoder position')
ax.set_ylabel('Attention weight')
ax.set_title('Bahdanau Attention Weights (one decoder step)')
ax.set_xticks(range(T_ENC))
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/nlp04_attention_bar.png', dpi=80)
plt.close()
print('\nAttention bar chart saved to /tmp/nlp04_attention_bar.png')


## Level 2: Seq2Seq with Bahdanau Attention in PyTorch

Task: reverse an integer sequence [1,2,3,4,5] → [5,4,3,2,1].

Architecture:
- **Encoder**: bidirectional GRU → encoder outputs [B, T, 2H]
- **BahdanauAttention**: computes context vector at each decoder step
- **Decoder**: GRU + attention, one step per output token

In [ ]:
# ── Seq2Seq dataset: reverse sequences of length 5 ──────────────────
VOCAB_SIZE_S2S = 10   # digits 0..9
SEQ_LEN_S2S    = 5
N_SAMPLES_S2S  = 500
EMBED_S2S      = 8
HIDDEN_S2S     = 16

def make_reversal_data(n: int, seq_len: int, vocab: int) -> tuple:
    """Generate (src, tgt) reversal pairs as long tensors."""
    src = np.random.randint(1, vocab, size=(n, seq_len))
    tgt = src[:, ::-1].copy()
    return (torch.tensor(src, dtype=torch.long),
            torch.tensor(tgt, dtype=torch.long))

X_rev, y_rev = make_reversal_data(N_SAMPLES_S2S, SEQ_LEN_S2S, VOCAB_SIZE_S2S)
split = 400
X_tr_rev, X_te_rev = X_rev[:split], X_rev[split:]
y_tr_rev, y_te_rev = y_rev[:split], y_rev[split:]
print(f'Train: {len(X_tr_rev)} samples, Test: {len(X_te_rev)} samples')

# ── Encoder ───────────────────────────────────────────────────────────
class Encoder(nn.Module):
    """Bidirectional GRU encoder; outputs [B, T, 2H] states."""
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn   = nn.GRU(embed_dim, hidden_dim,
                            batch_first=True, bidirectional=True)

    def forward(self, x: torch.Tensor) -> tuple:
        emb = self.embed(x)          # [B, T, E]
        out, h = self.rnn(emb)       # out: [B, T, 2H]; h: [2, B, H]
        # Merge bidirectional final states into [B, H] for decoder init
        h_merged = (h[0] + h[1])     # element-wise sum
        return out, h_merged

# ── Bahdanau Attention ────────────────────────────────────────────────
class BahdanauAttention(nn.Module):
    """Additive (Bahdanau) attention."""
    def __init__(self, hidden_dim: int):
        super().__init__()
        # W1 projects encoder keys [B, T, 2H] → [B, T, H]
        self.W1 = nn.Linear(hidden_dim * 2, hidden_dim)
        # W2 projects decoder query [B, H] → [B, H]
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        # v scores combined representation
        self.v  = nn.Linear(hidden_dim, 1)

    def forward(self, query: torch.Tensor,
                keys: torch.Tensor) -> tuple:
        """query: [B, H], keys: [B, T, 2H] -> context [B, 2H], alpha [B, T]"""
        q = self.W2(query).unsqueeze(1)   # [B, 1, H]
        k = self.W1(keys)                 # [B, T, H]
        score = self.v(torch.tanh(q + k)).squeeze(-1)  # [B, T]
        alpha = torch.softmax(score, dim=-1)            # [B, T]
        context = (alpha.unsqueeze(-1) * keys).sum(1)  # [B, 2H]
        return context, alpha

# ── Decoder with attention ─────────────────────────────────────────────
class AttentionDecoder(nn.Module):
    """GRU decoder with Bahdanau attention over encoder outputs."""
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed_dim)
        self.attn   = BahdanauAttention(hidden_dim)
        # Input to GRU = embed + context vector
        self.gru    = nn.GRU(embed_dim + hidden_dim * 2, hidden_dim,
                             batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward_step(self, prev_token: torch.Tensor,
                     h: torch.Tensor,
                     enc_out: torch.Tensor) -> tuple:
        """Single decoder step. Returns (logits [B, V], h_new [1, B, H], alpha [B, T])."""
        emb = self.embed(prev_token)          # [B, 1, E]
        context, alpha = self.attn(h.squeeze(0), enc_out)  # [B, 2H], [B, T]
        ctx_expanded = context.unsqueeze(1)   # [B, 1, 2H]
        gru_in = torch.cat([emb, ctx_expanded], dim=-1)    # [B, 1, E+2H]
        out, h_new = self.gru(gru_in, h)      # out: [B, 1, H]
        logits = self.fc_out(out.squeeze(1))  # [B, V]
        return logits, h_new, alpha

# ── Full seq2seq forward (teacher forcing) ────────────────────────────
def seq2seq_forward(encoder: nn.Module, decoder: nn.Module,
                    src: torch.Tensor, tgt: torch.Tensor,
                    teacher_forcing: bool = True) -> torch.Tensor:
    """Returns cross-entropy loss averaged over sequence positions."""
    enc_out, h = encoder(src)          # [B, T, 2H], [B, H]
    h = h.unsqueeze(0)                  # [1, B, H] for GRU
    B, T = tgt.shape
    # Start token: use tgt[:, 0] as the first decoder input
    dec_input = tgt[:, 0].unsqueeze(1)  # [B, 1]
    total_loss = torch.tensor(0.0, device=src.device)
    criterion  = nn.CrossEntropyLoss()

    for t in range(1, T):
        logits, h, _ = decoder.forward_step(dec_input, h, enc_out)
        total_loss  += criterion(logits, tgt[:, t])
        if teacher_forcing:
            dec_input = tgt[:, t].unsqueeze(1)    # ground truth next token
        else:
            dec_input = logits.argmax(-1).unsqueeze(1)  # model's own prediction

    return total_loss / (T - 1)

# ── Train 200 steps ───────────────────────────────────────────────────
enc_s2s = Encoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
dec_s2s = AttentionDecoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
params  = list(enc_s2s.parameters()) + list(dec_s2s.parameters())
opt_s2s = optim.Adam(params, lr=0.01)

BATCH = 32
losses_s2s = []
for step in range(200):
    idx = torch.randint(0, split, (BATCH,))
    src_b = X_tr_rev[idx].to(device)
    tgt_b = y_tr_rev[idx].to(device)
    opt_s2s.zero_grad()
    loss = seq2seq_forward(enc_s2s, dec_s2s, src_b, tgt_b, teacher_forcing=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    opt_s2s.step()
    losses_s2s.append(loss.item())

print(f'Final training loss: {losses_s2s[-1]:.4f}')

# ── Quick accuracy check ──────────────────────────────────────────────
def evaluate_s2s(encoder, decoder, X_te, y_te, batch_size=50):
    """Evaluate token-level accuracy on test set using greedy decoding."""
    encoder.eval(); decoder.eval()
    correct = total = 0
    with torch.no_grad():
        for i in range(0, len(X_te), batch_size):
            src_b = X_te[i:i+batch_size].to(device)
            tgt_b = y_te[i:i+batch_size].to(device)
            enc_out, h = encoder(src_b)
            h = h.unsqueeze(0)
            dec_in = tgt_b[:, 0].unsqueeze(1)
            for t in range(1, tgt_b.shape[1]):
                logits, h, _ = decoder.forward_step(dec_in, h, enc_out)
                pred = logits.argmax(-1)
                correct += (pred == tgt_b[:, t]).sum().item()
                total   += pred.shape[0]
                dec_in = pred.unsqueeze(1)
    encoder.train(); decoder.train()
    return correct / total if total > 0 else 0.0

acc_attn = evaluate_s2s(enc_s2s, dec_s2s, X_te_rev, y_te_rev)
print(f'Test token accuracy (with attention): {acc_attn:.3f}')


## Real-World Example 1: Attention Alignment Visualization

After training, we plot the attention weight matrix (decoder step vs encoder position)
as a heatmap. For the reversal task the ideal alignment should be anti-diagonal:
decoder step t should focus on encoder position (T-1-t).

In [ ]:
def get_attention_matrix(encoder: nn.Module, decoder: nn.Module,
                         src: torch.Tensor, tgt: torch.Tensor) -> np.ndarray:
    """Collect attention weights for one sample; returns [T_dec, T_enc]."""
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        src_b = src.unsqueeze(0).to(device)   # [1, T]
        tgt_b = tgt.unsqueeze(0).to(device)   # [1, T]
        enc_out, h = encoder(src_b)
        h = h.unsqueeze(0)
        dec_in = tgt_b[:, 0].unsqueeze(1)
        alphas = []
        for t in range(1, tgt_b.shape[1]):
            logits, h, alpha = decoder.forward_step(dec_in, h, enc_out)
            alphas.append(alpha.squeeze(0).cpu().numpy())  # [T_enc]
            dec_in = logits.argmax(-1).unsqueeze(1)
    encoder.train(); decoder.train()
    return np.array(alphas)  # [T_dec-1, T_enc]

# Pick a representative test sample
sample_src = X_te_rev[0]   # e.g. [3, 7, 2, 5, 1]
sample_tgt = y_te_rev[0]   # e.g. [1, 5, 2, 7, 3]  (reversed)

attn_matrix = get_attention_matrix(enc_s2s, dec_s2s, sample_src, sample_tgt)
print(f'Attention matrix shape: {attn_matrix.shape}  (dec_steps x enc_positions)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Heatmap: decoder step vs encoder position ─────────────────────────
ax = axes[0]
im = ax.imshow(attn_matrix, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xlabel('Encoder position (source)')
ax.set_ylabel('Decoder step (output)')
ax.set_title(f'Attention Alignment Heatmap\nSrc: {sample_src.tolist()}\nTgt: {sample_tgt.tolist()}')
ax.set_xticks(range(SEQ_LEN_S2S))
ax.set_yticks(range(SEQ_LEN_S2S - 1))
plt.colorbar(im, ax=ax)

# Annotate each cell with the weight
for i in range(attn_matrix.shape[0]):
    for j in range(attn_matrix.shape[1]):
        ax.text(j, i, f'{attn_matrix[i, j]:.2f}',
                ha='center', va='center', fontsize=8,
                color='black' if attn_matrix[i, j] < 0.6 else 'white')

# ── Training loss curve ───────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(losses_s2s, color='steelblue', alpha=0.8)
ax2.set_xlabel('Training step')
ax2.set_ylabel('Cross-entropy loss')
ax2.set_title('Seq2Seq with Attention — Training Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/nlp04_attn_heatmap.png', dpi=80, bbox_inches='tight')
plt.close()
print('Attention heatmap saved to /tmp/nlp04_attn_heatmap.png')
print('Ideal: attention peaks on anti-diagonal (decoder step t -> encoder pos T-1-t).')


## Real-World Example 2: No-Attention vs Attention on Longer Sequences

Attention matters more as sequence length grows. We compare a seq2seq model
without attention (only the encoder's final state is passed to the decoder)
against the attention model on sequences of length 5 and 10.

In [ ]:
class SimpleDecoder(nn.Module):
    """Decoder without attention — only receives encoder's final hidden state."""
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed_dim)
        self.gru    = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward_step(self, prev_token: torch.Tensor,
                     h: torch.Tensor) -> tuple:
        """Single step without attention."""
        emb = self.embed(prev_token)          # [B, 1, E]
        out, h_new = self.gru(emb, h)         # [B, 1, H]
        logits = self.fc_out(out.squeeze(1))  # [B, V]
        return logits, h_new

def train_noattn(seq_len: int, n_steps: int = 200, batch: int = 32) -> float:
    """Train no-attention seq2seq; return test token accuracy."""
    X_r, y_r = make_reversal_data(500, seq_len, VOCAB_SIZE_S2S)
    X_tr_r, X_te_r = X_r[:400], X_r[400:]
    y_tr_r, y_te_r = y_r[:400], y_r[400:]

    enc = Encoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
    dec = SimpleDecoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
    params_na = list(enc.parameters()) + list(dec.parameters())
    opt_na = optim.Adam(params_na, lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for step in range(n_steps):
        idx = torch.randint(0, 400, (batch,))
        src_b = X_tr_r[idx].to(device)
        tgt_b = y_tr_r[idx].to(device)
        opt_na.zero_grad()
        enc_out, h = enc(src_b)
        h = h.unsqueeze(0)
        dec_in = tgt_b[:, 0].unsqueeze(1)
        loss = torch.tensor(0.0, device=device)
        for t in range(1, seq_len):
            logits, h = dec.forward_step(dec_in, h)
            loss += criterion(logits, tgt_b[:, t])
            dec_in = tgt_b[:, t].unsqueeze(1)  # teacher forcing
        (loss / (seq_len - 1)).backward()
        torch.nn.utils.clip_grad_norm_(params_na, 1.0)
        opt_na.step()

    # Evaluate
    enc.eval(); dec.eval()
    correct = total = 0
    with torch.no_grad():
        src_te = X_te_r.to(device)
        tgt_te = y_te_r.to(device)
        _, h = enc(src_te)
        h = h.unsqueeze(0)
        dec_in = tgt_te[:, 0].unsqueeze(1)
        for t in range(1, seq_len):
            logits, h = dec.forward_step(dec_in, h)
            pred = logits.argmax(-1)
            correct += (pred == tgt_te[:, t]).sum().item()
            total   += pred.shape[0]
            dec_in  = pred.unsqueeze(1)
    return correct / total if total > 0 else 0.0

def train_attn(seq_len: int, n_steps: int = 200, batch: int = 32) -> float:
    """Train attention seq2seq on given seq_len; return test accuracy."""
    X_r, y_r = make_reversal_data(500, seq_len, VOCAB_SIZE_S2S)
    X_tr_r, X_te_r = X_r[:400], X_r[400:]
    y_tr_r, y_te_r = y_r[:400], y_r[400:]
    enc = Encoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
    dec = AttentionDecoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
    params_a = list(enc.parameters()) + list(dec.parameters())
    opt_a = optim.Adam(params_a, lr=0.01)
    for step in range(n_steps):
        idx = torch.randint(0, 400, (batch,))
        opt_a.zero_grad()
        loss = seq2seq_forward(enc, dec, X_tr_r[idx].to(device),
                               y_tr_r[idx].to(device), teacher_forcing=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params_a, 1.0)
        opt_a.step()
    return evaluate_s2s(enc, dec, X_te_r, y_te_r)

print('Comparing no-attention vs attention seq2seq:')
results_cmp = {}
for seq_len in [5, 10]:
    acc_na = train_noattn(seq_len)
    acc_at = train_attn(seq_len)
    results_cmp[seq_len] = (acc_na, acc_at)
    print(f'  seq_len={seq_len}: no-attn={acc_na:.3f}  attention={acc_at:.3f}')
print('\nAttention advantage typically grows with sequence length.')


## Real-World Example 3 + Comparison

### Example 3: Teacher Forcing vs Free Running
Teacher forcing uses ground-truth tokens as decoder inputs during training.
Free running uses the model's own predictions. The gap is called **exposure bias**.

### Comparison: No Attention vs Additive Attention vs Dot-Product Attention

In [ ]:
# ── Dot-product attention (for comparison) ───────────────────────────
class DotProductAttention(nn.Module):
    """Scaled dot-product attention (no learnable parameters)."""
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.scale = hidden_dim ** -0.5
        self.proj_k = nn.Linear(hidden_dim * 2, hidden_dim)  # project keys
        self.proj_q = nn.Linear(hidden_dim, hidden_dim)       # project query

    def forward(self, query: torch.Tensor,
                keys: torch.Tensor) -> tuple:
        """query: [B, H], keys: [B, T, 2H] -> context [B, 2H], alpha [B, T]"""
        q = self.proj_q(query).unsqueeze(1)   # [B, 1, H]
        k = self.proj_k(keys)                 # [B, T, H]
        score = torch.bmm(q, k.transpose(1, 2)).squeeze(1) * self.scale  # [B, T]
        alpha = torch.softmax(score, dim=-1)                               # [B, T]
        context = (alpha.unsqueeze(-1) * keys).sum(1)                     # [B, 2H]
        return context, alpha

class DotAttentionDecoder(nn.Module):
    """Decoder using dot-product attention."""
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed_dim)
        self.attn   = DotProductAttention(hidden_dim)
        self.gru    = nn.GRU(embed_dim + hidden_dim * 2, hidden_dim,
                             batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward_step(self, prev_token, h, enc_out):
        emb = self.embed(prev_token)            # [B, 1, E]
        context, alpha = self.attn(h.squeeze(0), enc_out)
        ctx_exp = context.unsqueeze(1)          # [B, 1, 2H]
        gru_in  = torch.cat([emb, ctx_exp], dim=-1)
        out, h_new = self.gru(gru_in, h)
        logits = self.fc_out(out.squeeze(1))
        return logits, h_new, alpha

# ── Teacher forcing ratio experiment ──────────────────────────────────
def train_with_tf_ratio(tf_ratio: float, n_steps: int = 200) -> float:
    """Train seq2seq with given teacher_forcing_ratio; return test accuracy."""
    enc = Encoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
    dec = AttentionDecoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
    params_tf = list(enc.parameters()) + list(dec.parameters())
    opt_tf = optim.Adam(params_tf, lr=0.01)
    criterion_tf = nn.CrossEntropyLoss()
    for step in range(n_steps):
        idx = torch.randint(0, 400, (32,))
        src_b = X_tr_rev[idx].to(device)
        tgt_b = y_tr_rev[idx].to(device)
        enc_out, h = enc(src_b)
        h = h.unsqueeze(0)
        dec_in = tgt_b[:, 0].unsqueeze(1)
        opt_tf.zero_grad()
        loss = torch.tensor(0.0, device=device)
        for t in range(1, SEQ_LEN_S2S):
            logits, h, _ = dec.forward_step(dec_in, h, enc_out)
            loss += criterion_tf(logits, tgt_b[:, t])
            # Scheduled sampling: use ground truth with probability tf_ratio
            use_gt = (torch.rand(1).item() < tf_ratio)
            dec_in = (tgt_b[:, t].unsqueeze(1) if use_gt
                      else logits.argmax(-1).unsqueeze(1))
        (loss / (SEQ_LEN_S2S - 1)).backward()
        torch.nn.utils.clip_grad_norm_(params_tf, 1.0)
        opt_tf.step()
    return evaluate_s2s(enc, dec, X_te_rev, y_te_rev)

print('Teacher forcing ratio experiment:')
tf_ratios = [1.0, 0.5, 0.0]
tf_accs = {}
for tf_r in tf_ratios:
    acc_tf = train_with_tf_ratio(tf_r)
    tf_accs[tf_r] = acc_tf
    print(f'  tf_ratio={tf_r:.1f}: test accuracy={acc_tf:.3f}')

# ── Train dot-product attention for comparison ─────────────────────────
enc_dot = Encoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
dec_dot = DotAttentionDecoder(VOCAB_SIZE_S2S, EMBED_S2S, HIDDEN_S2S).to(device)
params_dot = list(enc_dot.parameters()) + list(dec_dot.parameters())
opt_dot = optim.Adam(params_dot, lr=0.01)
for step in range(200):
    idx = torch.randint(0, split, (32,))
    opt_dot.zero_grad()
    loss = seq2seq_forward(enc_dot, dec_dot, X_tr_rev[idx].to(device),
                           y_tr_rev[idx].to(device), teacher_forcing=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params_dot, 1.0)
    opt_dot.step()
acc_dot = evaluate_s2s(enc_dot, dec_dot, X_te_rev, y_te_rev)

print(f'\nAttention comparison on reversal task (seq_len=5):')
print(f'  No attention:           {results_cmp.get(5, (0,0))[0]:.3f}')
print(f'  Additive (Bahdanau):    {acc_attn:.3f}')
print(f'  Dot-product (scaled):   {acc_dot:.3f}')

# ── Final comparison bar chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
tf_labels = [f'TF={r:.1f}' for r in tf_ratios]
tf_vals   = [tf_accs[r] for r in tf_ratios]
bars = ax.bar(tf_labels, tf_vals, color=['steelblue', 'darkorange', 'forestgreen'])
for bar, val in zip(bars, tf_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Test token accuracy')
ax.set_title('Teacher Forcing Ratio vs Accuracy\n(Exposure Bias Effect)')
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
attn_labels = ['No attention', 'Additive\n(Bahdanau)', 'Dot-product']
attn_vals   = [results_cmp.get(5, (0,0))[0], acc_attn, acc_dot]
attn_colors = ['#d62728', 'steelblue', 'darkorange']
bars2 = ax2.bar(attn_labels, attn_vals, color=attn_colors)
for bar, val in zip(bars2, attn_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax2.set_ylabel('Test token accuracy')
ax2.set_title('No Attention vs Additive vs Dot-Product\n(Seq len=5)')
ax2.set_ylim([0, 1.1])
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Seq2Seq Attention Mechanism Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/nlp04_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print('\nFinal comparison plots saved to /tmp/nlp04_comparison.png')
print('\nKey insight: both additive and dot-product attention outperform')
print('no-attention seq2seq. Teacher forcing (TF=1.0) usually converges faster')
print('but TF=0.5 generalises better due to reduced exposure bias.')
